# Cyclistic Bike-Share — Analyze Phase

Business question: **how do annual members and casual riders use Cyclistic bikes differently?**

Uses the cleaned dataset produced in the Process phase
(`data/processed/all_trips_2025.parquet`, 5,400,008 rows). Small aggregate tables produced here
are saved to `data/summary/` for reuse in the Share phase.

## Setup

In [1]:
import pandas as pd
import os

trips = pd.read_parquet("../data/processed/all_trips_2025.parquet")
SUMMARY_DIR = "../data/summary"
os.makedirs(SUMMARY_DIR, exist_ok=True)
DAY_ORDER = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
print(f"Loaded {len(trips)} rows")

Loaded 5400008 rows

## 1. Ride length — mean, median, max by rider type

The single clearest signal in the data: casual riders take **substantially longer rides**.

In [2]:
overall = trips.groupby("member_casual")["ride_length_min"].agg(["count", "mean", "median", "max"]).round(2)
overall.to_csv(os.path.join(SUMMARY_DIR, "overall_ride_length_stats.csv"))
overall

                 count   mean  median      max
member_casual
casual         1915806  19.91   11.89  1439.98
member         3484202  12.18    8.74  1439.90

**Casual riders ride ~63% longer on average** (19.91 vs 12.18 min) and ~36% longer at the median (11.89 vs 8.74 min).

## 2. Mode of day_of_week

In [3]:
print("Overall mode day:", trips["day_name"].mode()[0])
trips.groupby("member_casual")["day_name"].agg(lambda s: s.mode()[0])

Overall mode day: Saturday
member_casual
casual    Saturday
member    Thursday
Name: day_name, dtype: str

Casual riders' single busiest day is **Saturday**; members' is **Thursday** — an early signal of leisure vs. commute usage.

## 3. Rides and average duration by day of week

In [4]:
by_day = trips.groupby(["day_name", "member_casual"], observed=True).agg(
    ride_count=("ride_id", "count"),
    avg_ride_length_min=("ride_length_min", "mean"),
).reset_index()
by_day["day_name"] = pd.Categorical(by_day["day_name"], categories=DAY_ORDER, ordered=True)
by_day = by_day.sort_values(["day_name", "member_casual"])
by_day["avg_ride_length_min"] = by_day["avg_ride_length_min"].round(2)
by_day.to_csv(os.path.join(SUMMARY_DIR, "rides_by_day_of_week.csv"), index=False)
by_day

 day_name member_casual  ride_count  avg_ride_length_min
   Sunday        casual      316953                23.10
   Sunday        member      374535                13.45
   Monday        casual      219232                19.70
   Monday        member      493322                11.76
  Tuesday        casual      216926                17.62
  Tuesday        member      552529                11.85
Wednesday        casual      212744                16.36
Wednesday        member      540212                11.63
 Thursday        casual      247893                17.45
 Thursday        member      565177                11.73
   Friday        casual      306463                19.60
   Friday        member      518551                12.14
 Saturday        casual      395595                22.40
 Saturday        member      439876                13.30

**Members** show a clear weekday pattern: ride *counts* peak Tue–Thu (commuting) and stay
fairly flat, with duration barely moving (11.6–13.5 min all week). **Casual riders** show the
opposite: counts and duration both climb toward the weekend, peaking Saturday (23.10 min avg,
the longest of any day for either group) — consistent with leisure trips.

## 4. Rides and average duration by month (seasonality)

In [5]:
by_month = trips.groupby(["month", "member_casual"], observed=True).agg(
    ride_count=("ride_id", "count"),
    avg_ride_length_min=("ride_length_min", "mean"),
).reset_index()
by_month["avg_ride_length_min"] = by_month["avg_ride_length_min"].round(2)
by_month.to_csv(os.path.join(SUMMARY_DIR, "rides_by_month.csv"), index=False)
by_month

 month member_casual  ride_count  avg_ride_length_min
     1        casual       23405                11.97
     1        member      112331                 9.98
     2        casual       27003                12.38
     2        member      122097                 9.96
     3        casual       82864                17.89
     3        member      208458                11.13
     4        casual      105260                18.66
     4        member      257921                11.27
     5        casual      175655                20.83
     5        member      313996                11.90
     6        casual      278702                21.91
     6        member      379523                12.85
     7        casual      308446                21.44
     7        member      430394                13.15
     8        casual      323533                21.71
     8        member      443130                13.08
     9        casual      254727                19.55
     9        member      44

Both rider types ride most in summer (Jun–Aug) and least in winter (Jan–Feb), but **casual
ridership is far more seasonal**: casual ride counts grow ~14x from January (23,405) to August
(323,533), while member counts grow only ~4x (112,331 → 443,130) over the same span. Members ride
fairly consistently year-round; casual riders are much more weather/season-dependent.

## 5. Rides by hour of day

In [6]:
by_hour = trips.groupby(["hour", "member_casual"], observed=True).agg(
    ride_count=("ride_id", "count"),
).reset_index()
by_hour.to_csv(os.path.join(SUMMARY_DIR, "rides_by_hour.csv"), index=False)
by_hour.pivot(index="hour", columns="member_casual", values="ride_count")

member_casual  casual  member
hour
0               36762   31391
1               23492   19107
2               15667   11430
3                8694    7520
4                6941    8606
5               11009   33525
6               25696   99153
7               47925  196413
8               68010  252034
9               68273  163882
10              82449  141036
11             105312  165364
12             123711  186824
13             126735  181968
14             133239  184831
15             148055  232972
16             168789  328673
17             183262  375149
18             156921  290116
19             116407  200336
20              83803  138362
21              71204  107972
22              59981   78612
23              43469   48926

**Members show a textbook commuting pattern**: a sharp morning peak at 7–8 AM and an even
larger evening peak at 16–18h (4–6 PM), with a trough mid-morning. **Casual riders have no sharp
AM peak** — their volume ramps up gradually through the morning and peaks in the afternoon
(12–18h), consistent with leisure rather than commute trips.

## 6. Bike type preference

In [7]:
by_bike = trips.groupby(["member_casual", "rideable_type"], observed=True).size().unstack()
by_bike_pct = (by_bike.div(by_bike.sum(axis=1), axis=0) * 100).round(1)
by_bike.to_csv(os.path.join(SUMMARY_DIR, "bike_type_by_member.csv"))
print(by_bike)
print()
print(by_bike_pct)

rideable_type  classic_bike  electric_bike
member_casual
casual               667993        1247813
member              1274451        2209751

rideable_type  classic_bike  electric_bike
member_casual
casual                 34.9           65.1
member                 36.6           63.4

Both groups prefer electric bikes (~63–65%) over classic; the gap between rider types here is small and not a strong differentiator.

## 7. Weekday vs. weekend split

In [8]:
trips["is_weekend"] = trips["day_name"].isin(["Saturday", "Sunday"])
weekend = trips.groupby(["member_casual", "is_weekend"], observed=True).size().unstack()
weekend.columns = ["weekday", "weekend"]
weekend_pct = (weekend.div(weekend.sum(axis=1), axis=0) * 100).round(1)
weekend.to_csv(os.path.join(SUMMARY_DIR, "weekday_vs_weekend.csv"))
print(weekend)
print()
print(weekend_pct)

               weekday  weekend
member_casual
casual         1203258   712548
member         2669791   814411

               weekday  weekend
member_casual
casual            62.8     37.2
member            76.6     23.4

**Members ride 76.6% on weekdays vs. 23.4% on weekends — a strong commute signature.**
Casual riders are still majority-weekday (62.8%) since weekdays are 5/7 of the week, but their
weekend share (37.2%) is **59% relatively higher** than members' (23.4%), confirming casual usage
skews leisure/weekend.

## Summary of key findings

1. **Ride length**: casual riders ride ~63% longer on average (19.9 vs 12.2 min) and this gap is
   consistent across every day of the week.
2. **Weekly pattern**: members ride steadily Mon–Fri with a clear commute rhythm (counts peak
   Tue–Thu); casual riders ride progressively more toward the weekend, peaking Saturday in both
   volume and duration.
3. **Seasonality**: casual ridership is much more seasonal (~14x growth from Jan to Aug) than
   member ridership (~4x) — casual usage is weather/leisure-driven, member usage is more
   utilitarian and stable year-round.
4. **Time of day**: members show a classic two-peak commuting profile (AM 7–8h, PM 16–18h);
   casual riders show a single broad afternoon peak with no morning rush — consistent with
   leisure trips rather than commuting.
5. **Weekday/weekend split**: members are 76.6% weekday rides; casual riders' weekend share
   (37.2%) is proportionally much higher, reinforcing the leisure-vs-commute divide.
6. **Bike type**: not a strong differentiator — both groups prefer electric bikes at similar
   rates (63–65%).

**Overall**: the data supports a clear behavioral split — members use Cyclistic primarily for
**commuting** (short, frequent, weekday, rush-hour rides, stable year-round), while casual riders
use it primarily for **leisure** (longer, weekend- and summer-skewed, midday/afternoon rides).
This distinction will directly inform the marketing recommendations in the Act phase.